**INSTRUCCIONES DE ENTREGA**

1. Completa la función limpiar_resena(texto).
2. Ejecuta tu función con las 3 reseñas de prueba.
3  Entrega:

* **a) Código:** Tu función implementada.
* **b) Salida:** El resultado obtenido para cada reseña.
* **c) Análisis:** Una breve explicación respondiendo:
    * ¿Qué pasos aplicaste?
    * ¿Por qué conservaste **"no"**, **"ni"** y **"nunca"**?
    * ¿Qué ventaja tiene convertir emojis a texto?
    * ¿En qué tipo de aplicación real usarías este preprocesamiento?

# Librería 

In [3]:
# 'nltk' (Natural Language Toolkit) es la librería líder para trabajar con datos de lenguaje humano en Python.
import nltk

# 're' es el módulo de Expresiones Regulares; sirve para buscar, extraer o reemplazar patrones de texto
# (como URLs, etiquetas HTML o caracteres especiales).
import re

# 'string' es un módulo estándar de Python que usamos aquí principalmente para obtener 
# una lista rápida de todos los signos de puntuación (string.punctuation).
import string

# 'word_tokenize' es una función que divide una cadena de texto en unidades mínimas llamadas "tokens" 
# (generalmente palabras y signos de puntuación individuales).
from nltk.tokenize import word_tokenize

# 'stopwords' son palabras muy comunes (como "la", "un", "de", "con") que generalmente 
# no aportan significado semántico y se eliminan para limpiar el texto.
from nltk.corpus import stopwords

# 'SnowballStemmer' es un algoritmo que reduce las palabras a su raíz o "stem" 
# (por ejemplo: "comiendo", "comió" y "comer" se reducen a "com"). Ayuda a normalizar el vocabulario.
from nltk.stem import SnowballStemmer

# Comentó estas líneas porque los recursos ya fueron descargados localmente.

# Descarga el tokenizador 'punkt'. Es el modelo preentrenado que utiliza NLTK 
# para saber dónde termina una palabra y dónde empieza otra, reconociendo abreviaturas y puntos.
#nltk.download('punkt')

# Descarga datos adicionales de tabulación y estructuras para el tokenizador. 
# Es necesario en versiones recientes de NLTK para procesar correctamente los tokens en ciertos idiomas.
#nltk.download('punkt_tab')

# Descarga las listas de "palabras vacías" (stopwords) para múltiples idiomas. 
# Sin esto, NLTK no sabría qué palabras filtrar cuando pides stopwords.words('spanish').
#nltk.download('stopwords')

## Funciones

In [5]:
# Definimos una función para convertir emojis en texto descriptivo.
# Esto es vital porque los modelos de IA suelen ignorar los símbolos gráficos.
def reemplazar_emojis(texto):
    # Creamos un diccionario (mapa) que vincula cada emoji con una etiqueta de texto clara.
    # Agregamos espacios alrededor de la etiqueta para que no se pegue a otras palabras.
    mapa_emojis = {
        "😡": " emoji_enojo ",
        "😠": " emoji_molestia ",
        "😊": " emoji_felicidad ",
        "😀": " emoji_alegria ",
        "😍": " emoji_amor ",
        "😢": " emoji_tristeza ",
        "😭": " emoji_llanto ",
        "😱": " emoji_sorpresa ",
        "🤢": " emoji_asco ",
        "🍕": " emoji_pizza ",
        "🐶": " emoji_perro "
    }

    # Recorremos el diccionario pareja por pareja (emoji y su etiqueta descriptiva).
    for emoji, etiqueta in mapa_emojis.items():
        # Reemplazamos cada ocurrencia del emoji visual por su nombre de etiqueta.
        texto = texto.replace(emoji, etiqueta)

    # Devolvemos el texto ya "traducido" para que el modelo pueda "leer" las emociones.
    return texto

In [38]:
def limpiar_resena(texto, verbose=0):
    if verbose: print(f"Pasos de la función: Limpiar_resena by Carlos Santos \n\n[0] Texto original: {texto}") # Texto original
    # 1. Quitar HTML: Busca cualquier cosa entre '<' y '>' y la borra.
    # Evita que etiquetas como <b> o <div> ensucien el contenido real.
    texto = re.sub(r'<.*?>', '', texto)
    if verbose: print(f"[1] Quitar HTML: {texto}")

    # 2. Quitar URLs: Elimina enlaces que empiecen con http o www.
    # En una reseña, la URL no suele aportar significado emocional.
    texto = re.sub(r'http\S+|www\S+', '', texto)
    if verbose: print(f"[2] Quitar URLs: {texto}")

    # 3. Pasar a minúsculas: Normaliza el texto ("Pizza" y "pizza" son distintas).
    texto = texto.lower()
    if verbose: print(f"[3] Pasar a minúsculas: {texto}")

    # 4. Reemplazar emojis: Llama a nuestra función previa para darles nombre de texto.
    texto = reemplazar_emojis(texto)
    if verbose: print(f"[4] Reemplazar emojis: {texto}")

    # 5. Quitar símbolos raros: Usa una expresión regular para dejar solo letras, espacios y tildes.
    # El símbolo '^' dentro de los corchetes significa "todo lo que NO sea esto".
    # Elimina símbolos no deseados del texto
    # [^\w\sáéíóúüñ¡!¿?.,] significa:
    # ^        → negación (todo lo que NO esté en el conjunto)
    # \w       → letras y números (incluye _)
    # \s       → espacios
    # áéíóúüñ  → caracteres propios del español
    # ¡!¿?.,   → signos de puntuación que queremos conservar
    #
    # Todo lo que NO cumpla con lo anterior se reemplaza por un espacio
    texto = re.sub(r'[^\w\sáéíóúüñ¡!¿?.,]', ' ', texto)
    if verbose: print(f"[5] Quitar símbolos raros: {texto}")

    # 6. Quitar puntuación: Limpia puntos, comas, signos de exclamación, etc.
    # Usamos una tabla de traducción para borrarlos de forma eficiente.
    # Se utiliza el método translate() para eliminar caracteres específicos del texto.
    # str.maketrans('', '', caracteres) crea una tabla de traducción donde:
    # - '' (primer argumento) → no se reemplaza nada
    # - '' (segundo argumento) → no se sustituye ningún carácter
    # - caracteres (tercer argumento) → se eliminan del texto
    
    # string.punctuation incluye signos como: . , ! ? ; : etc.
    # Se agregan manualmente '¡¿' porque no vienen incluidos por defecto  y son comunes en español

    texto = texto.translate(str.maketrans('', '', string.punctuation + '¡¿'))
    if verbose: print(f"[6] Quitar puntuación: {texto}")

    # 7. Quitar números: Borra cualquier dígito del 0 al 9.
    texto = re.sub(r'\d+', '', texto)
    if verbose: print(f"[7] Quitar números: {texto}")

    # 8. Quitar espacios extra: Convierte múltiples espacios seguidos en uno solo y limpia los bordes.
    texto = re.sub(r'\s+', ' ', texto).strip()
    if verbose: print(f"[8] Quitar espacios extra: {texto}")

    # 9. Tokenizar: Divide la frase en una lista de palabras individuales usando reglas del español.
    tokens = word_tokenize(texto, language='spanish')
    if verbose: print(f"[9] Tokenizar: {tokens}")

    # 10. Quitar stopwords conservando negaciones: 
    # Creamos un set de palabras irrelevantes pero sacamos de ahí las que cambian el sentido.
    stop_words = set(stopwords.words('spanish'))
    palabras_negacion = {'no', 'ni', 'nunca'}
    # Filtramos la lista: solo nos quedamos con las palabras que NO están en la lista de stopwords.
    tokens = [t for t in tokens if t in palabras_negacion or t not in stop_words]
    if verbose: print(f"[10] Quitar stopwords: {tokens}")

    # 11. Aplicar stemming: Reduce cada palabra a su raíz para agrupar significados.
    # Ej: "comiendo" y "comió" se convierten en "com".
    stemmer = SnowballStemmer('spanish')
    tokens = [stemmer.stem(t) for t in tokens]
    if verbose: print(f"[11] Aplicar stemming: {tokens}\n")

    # Devolvemos la lista de tokens finales listos para ser analizados.
    return tokens

In [40]:

# 4. Textos de prueba
resena_1 = "La pizza estaba <b>INCREÍBLE</b>. 🍕 Llegó en 20 minutos. http://pizza.com"
resena_2 = "¿Alguien sabe si abren el domingo? Quiero llevar a mis 2 perros. 🐶"
resena_3 = "No me gustó el servicio, pero la decoración no estaba mal."

# 6. Pruebas
# Descomenta estas líneas cuando termines tu función:
print(limpiar_resena(resena_1,1))


Pasos de la función: Limpiar_resena by Carlos Santos 

[0] Texto original: La pizza estaba <b>INCREÍBLE</b>. 🍕 Llegó en 20 minutos. http://pizza.com
[1] Quitar HTML: La pizza estaba INCREÍBLE. 🍕 Llegó en 20 minutos. http://pizza.com
[2] Quitar URLs: La pizza estaba INCREÍBLE. 🍕 Llegó en 20 minutos. 
[3] Pasar a minúsculas: la pizza estaba increíble. 🍕 llegó en 20 minutos. 
[4] Reemplazar emojis: la pizza estaba increíble.  emoji_pizza  llegó en 20 minutos. 
[5] Quitar símbolos raros: la pizza estaba increíble.  emoji_pizza  llegó en 20 minutos. 
[6] Quitar puntuación: la pizza estaba increíble  emojipizza  llegó en 20 minutos 
[7] Quitar números: la pizza estaba increíble  emojipizza  llegó en  minutos 
[8] Quitar espacios extra: la pizza estaba increíble emojipizza llegó en minutos
[9] Tokenizar: ['la', 'pizza', 'estaba', 'increíble', 'emojipizza', 'llegó', 'en', 'minutos']
[10] Quitar stopwords: ['pizza', 'increíble', 'emojipizza', 'llegó', 'minutos']
[11] Aplicar stemming: ['pizz', 

In [42]:
print(limpiar_resena(resena_2,1))

Pasos de la función: Limpiar_resena by Carlos Santos 

[0] Texto original: ¿Alguien sabe si abren el domingo? Quiero llevar a mis 2 perros. 🐶
[1] Quitar HTML: ¿Alguien sabe si abren el domingo? Quiero llevar a mis 2 perros. 🐶
[2] Quitar URLs: ¿Alguien sabe si abren el domingo? Quiero llevar a mis 2 perros. 🐶
[3] Pasar a minúsculas: ¿alguien sabe si abren el domingo? quiero llevar a mis 2 perros. 🐶
[4] Reemplazar emojis: ¿alguien sabe si abren el domingo? quiero llevar a mis 2 perros.  emoji_perro 
[5] Quitar símbolos raros: ¿alguien sabe si abren el domingo? quiero llevar a mis 2 perros.  emoji_perro 
[6] Quitar puntuación: alguien sabe si abren el domingo quiero llevar a mis 2 perros  emojiperro 
[7] Quitar números: alguien sabe si abren el domingo quiero llevar a mis  perros  emojiperro 
[8] Quitar espacios extra: alguien sabe si abren el domingo quiero llevar a mis perros emojiperro
[9] Tokenizar: ['alguien', 'sabe', 'si', 'abren', 'el', 'domingo', 'quiero', 'llevar', 'a', 'mis', 'p

In [44]:
print(limpiar_resena(resena_3,1))

Pasos de la función: Limpiar_resena by Carlos Santos 

[0] Texto original: No me gustó el servicio, pero la decoración no estaba mal.
[1] Quitar HTML: No me gustó el servicio, pero la decoración no estaba mal.
[2] Quitar URLs: No me gustó el servicio, pero la decoración no estaba mal.
[3] Pasar a minúsculas: no me gustó el servicio, pero la decoración no estaba mal.
[4] Reemplazar emojis: no me gustó el servicio, pero la decoración no estaba mal.
[5] Quitar símbolos raros: no me gustó el servicio, pero la decoración no estaba mal.
[6] Quitar puntuación: no me gustó el servicio pero la decoración no estaba mal
[7] Quitar números: no me gustó el servicio pero la decoración no estaba mal
[8] Quitar espacios extra: no me gustó el servicio pero la decoración no estaba mal
[9] Tokenizar: ['no', 'me', 'gustó', 'el', 'servicio', 'pero', 'la', 'decoración', 'no', 'estaba', 'mal']
[10] Quitar stopwords: ['no', 'gustó', 'servicio', 'decoración', 'no', 'mal']
[11] Aplicar stemming: ['no', 'gust', 

**INSTRUCCIONES DE ENTREGA**

1. Completa la función limpiar_resena(texto).
2. Ejecuta tu función con las 3 reseñas de prueba.
3  Entrega:

* **a) Código:** Tu función implementada.
* **b) Salida:** El resultado obtenido para cada reseña.
* **c) Análisis:** Una breve explicación respondiendo:
    * ¿Qué pasos aplicaste?
        1.  **Limpieza de ruido:** Eliminación de etiquetas HTML y URLs.
        2.  **Normalización de texto:** Conversión a minúsculas, eliminación de números y espacios extra.
        3.  **Tratamiento semántico:** Reemplazo de emojis por texto y preservación de negaciones.
        4.  **Simplificación léxica:** Eliminación de signos de puntuación, filtrado de **stopwords** y aplicación de **Stemming**  para agrupar palabras con el mismo significado. 
      
    * ¿Por qué conservaste **"no"**, **"ni"** y **"nunca"**?
 
        En el **análisis de sentimientos**, estas palabras son fundamentales porque actúan como **modificadores de polaridad**. Si las eliminamos como si fueran **stopwords** comunes, una frase como *"No hay becas en el IPN"* se convertiría simplemente en *"bueno"*, cambiando totalmente el sentido de la reseña de negativo a positivo.

    * ¿Qué ventaja tiene convertir emojis a texto?

    Los emojis cargan una **fuerte carga emocional** y semántica. Al convertirlos a texto (por ejemplo, de 🍕 a "pizza" o de 😡 a "enojado"):

        * Evitamos perder información valiosa para el modelo.
        * Permitimos que el algoritmo procese el sentimiento del emoji igual que una palabra escrita.
        * Reducimos la dimensionalidad del vocabulario al estandarizar iconos visuales en términos textuales conocidos.
      
    * ¿En qué tipo de aplicación real usarías este preprocesamiento?

        Es ideal para:
      
        * **Análisis de satisfacción del cliente:** Procesar automáticamente miles de quejas o comentarios en redes sociales (Twitter/X, Facebook) para detectar crisis de marca.
        * **Clasificación de reseñas en E-commerce:** Para que plataformas como Amazon o Mercado Libre puedan categorizar automáticamente si una experiencia de compra fue positiva o negativa a gran escala. 
        * **Chatbots de servicio técnico:** Para entender la intención y el sentimiento del usuario antes de derivarlo a un agente humano.
        *    En mi caso particular, es directamente aplicable en mi actual trabjo en La Comer para optimizar el procesamiento de los altos volúmenes de comentarios y quejas de clientes.